# 8. Monte Carlo with Exploring Starts (MC ES)
### Ensures Every State-Action Pair is Explored

**Monte Carlo with Exploring Starts (MC ES)** solves the exploration challenge by assuming that every episode begins at a state-action pair $(S_0, A_0)$ chosen uniformly at random from all possible states and actions:

$$P(S_0 = s, A_0 = a) > 0 \quad \forall s \in \mathcal{S}, a \in \mathcal{A}(s)$$

#### Key Concepts:
1. **Guaranteed Complete Exploration**:
   Because every pair $(s, a)$ has positive probability of being the initial step, all state-action values $Q(s, a)$ are guaranteed to be updated infinitely often as episodes approach infinity.
2. **Pure Greedy Policy Updates**:
   No $\epsilon$-greediness is required during the trajectory after step 0! The policy can remain purely deterministic:
   $$\pi(s) = \arg\max_a Q(s, a)$$
3. **Exact Optimal Control**:
   Avoids the suboptimality gap introduced by $\epsilon$-greedy exploration policies.


In [ ]:
# Step 1: Import Libraries
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict


In [ ]:
# Step 2: Define a Pure Python Blackjack Environment (Sutton & Barto Ch. 5)
# State: (player_sum: 12..21, dealer_showing_card: 1..10, usable_ace: bool)
# Actions: 0 = Stick, 1 = Hit

class BlackjackEnv:
    def __init__(self):
        self.actions = [0, 1]  # 0: Stick, 1: Hit
        
    def _draw_card(self):
        card = np.random.randint(1, 14)
        return min(card, 10)  # Face cards (J, Q, K) have value 10; Ace is 1
        
    def _draw_hand(self):
        return [self._draw_card(), self._draw_card()]
        
    def _get_hand_value(self, hand):
        val = sum(hand)
        has_ace = 1 in hand
        if has_ace and val + 10 <= 21:
            return val + 10, True  # Usable Ace
        return val, False
        
    def reset(self, start_state=None, start_action=None):
        if start_state is not None:
            player_sum, dealer_card, usable_ace = start_state
            self.player_hand = [1, player_sum - 11] if usable_ace else [player_sum // 2, player_sum - player_sum // 2]
            self.dealer_hand = [dealer_card, self._draw_card()]
        else:
            self.player_hand = self._draw_hand()
            while self._get_hand_value(self.player_hand)[0] < 12:
                self.player_hand.append(self._draw_card())
            self.dealer_hand = self._draw_hand()
            
        p_val, usable_ace = self._get_hand_value(self.player_hand)
        return (p_val, self.dealer_hand[0], usable_ace)
        
    def step(self, action):
        if action == 1:  # Hit
            self.player_hand.append(self._draw_card())
            p_val, usable_ace = self._get_hand_value(self.player_hand)
            if p_val > 21:
                return (p_val, self.dealer_hand[0], usable_ace), -1.0, True  # Bust
            return (p_val, self.dealer_hand[0], usable_ace), 0.0, False
        else:  # Stick: Dealer plays
            while True:
                d_val, _ = self._get_hand_value(self.dealer_hand)
                if d_val >= 17:
                    break
                self.dealer_hand.append(self._draw_card())
                
            p_val, usable_ace = self._get_hand_value(self.player_hand)
            d_val, _ = self._get_hand_value(self.dealer_hand)
            
            if d_val > 21 or p_val > d_val:
                reward = 1.0  # Player wins
            elif p_val == d_val:
                reward = 0.0  # Draw
            else:
                reward = -1.0 # Dealer wins
                
            return (p_val, self.dealer_hand[0], usable_ace), reward, True

env = BlackjackEnv()
print("Blackjack environment initialized.")


In [ ]:
# Step 3: Implement Monte Carlo Exploring Starts (MC ES)
def monte_carlo_es(env, num_episodes=100000):
    Q = defaultdict(lambda: np.zeros(len(env.actions)))
    returns_sum = defaultdict(float)
    returns_count = defaultdict(int)
    
    # Initialize deterministic policy
    pi = defaultdict(int)
    
    for ep in range(1, num_episodes + 1):
        # Exploring Start: randomly choose S0 and A0
        start_player = np.random.randint(12, 22)
        start_dealer = np.random.randint(1, 11)
        start_ace = bool(np.random.choice([True, False]))
        start_state = (start_player, start_dealer, start_ace)
        start_action = np.random.choice(env.actions)
        
        # Reset with exploring start
        s = env.reset(start_state=start_state)
        a = start_action
        
        episode = []
        # First step from exploring start
        next_s, r, done = env.step(a)
        episode.append((s, a, r))
        s = next_s
        
        # Follow current greedy policy π for subsequent steps
        while not done:
            a = pi[s] # Pure greedy action
            next_s, r, done = env.step(a)
            episode.append((s, a, r))
            s = next_s
            
        # First-visit update of Q
        G = 0.0
        for t in reversed(range(len(episode))):
            s_t, a_t, r_t = episode[t]
            G = 1.0 * G + r_t
            
            if (s_t, a_t) not in [(step[0], step[1]) for step in episode[:t]]:
                returns_sum[(s_t, a_t)] += G
                returns_count[(s_t, a_t)] += 1
                Q[s_t][a_t] = returns_sum[(s_t, a_t)] / returns_count[(s_t, a_t)]
                # Policy improvement: update greedy policy
                pi[s_t] = np.argmax(Q[s_t])
                
    return Q, pi

Q_es, pi_es = monte_carlo_es(env, num_episodes=100000)
print(f"MC ES completed. Evaluated {len(pi_es)} unique states.")


In [ ]:
# Step 4: Visualize the Optimal Policy Surfaces Derived via Exploring Starts
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, usable_ace, title in [(ax1, False, 'No Usable Ace'), (ax2, True, 'Usable Ace')]:
    grid = np.zeros((10, 10))
    for p_i, p in enumerate(range(12, 22)):
        for d_i, d in enumerate(range(1, 11)):
            grid[p_i, d_i] = pi_es.get((p, d, usable_ace), 0)
            
    im = ax.imshow(grid, cmap='coolwarm', origin='lower')
    ax.set_xticks(range(10))
    ax.set_xticklabels(range(1, 11))
    ax.set_yticks(range(10))
    ax.set_yticklabels(range(12, 22))
    ax.set_xlabel("Dealer Showing Card")
    ax.set_ylabel("Player Sum")
    ax.set_title(f"Optimal Policy π* ({title})")

plt.colorbar(im, ax=[ax1, ax2], ticks=[0, 1], label='Action (0: Stick, 1: Hit)')
plt.show()


### Summary
- Guarantees complete state-action space exploration by randomly seeding starting states and actions.
- Allows the policy to be purely deterministic and greedy without sacrificing exploration.
